# Transformer Risk Dashboard
### Last updated: 2026-08-17 by Bridget Knight (bknight@mhdld.com)
---
This tool helps identify transformers that are at risk of overloading during hot weather periods. It is designed to run locally <br>
on an MMLD workstation to ensure a secure, trusted connection to the internal database server (`MMLDAPP03`). Please <br>see the `transformers_analysis_user_guide` document that is packaged with this notebook for further instructions.

In [ ]:
# SETUP & DEPENDENCY INSTALLER
# ==========================================
import sys
import subprocess

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing missing package: {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Ensure core packages are present
install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("plotly")
install_if_missing("ipywidgets")
install_if_missing("pyodbc")
install_if_missing("pyproj")
install_if_missing("openmeteo-requests", "openmeteo_requests")
install_if_missing("requests-cache", "requests_cache")
install_if_missing("retry-requests", "retry_requests")

# --- DATA ---
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# --- SQL ---
import pyodbc
from binascii import hexlify
from pyproj import Transformer

# --- WEATHER ---
import openmeteo_requests
import requests_cache
from retry_requests import retry

# --- PLOTTING ---
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.express as px
pio.renderers.default = "notebook_connected"

# --- WIDGETS ---
from ipywidgets import widgets, Layout
from IPython.display import clear_output, HTML, display

# --- OTHER ---
import warnings
warnings.filterwarnings("ignore")
from functools import wraps
import traceback

print("All dependencies checked and loaded successfully!")

All dependencies checked and loaded successfully!


In [ ]:
# FUNCTIONS
# ==========================================
def connect_to_db(server="MMLDAPP03", database="GridAnalysis", driver="{ODBC Driver 17 for SQL Server}"):

    conn_str = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        "Trusted_Connection=yes;"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )

    conn = pyodbc.connect(conn_str, timeout=10)
    #cursor = conn.cursor()
    return conn

def _handle_geometry(geometry_value):
    return f"0x{hexlify(geometry_value).decode().upper()}"

def get_meters_xfmrs(conn):
    cursor = conn.cursor()
    meters_xfmr_query = """
    SELECT [METERID]
        ,[METERLATITUDE]
        ,[METERLONGITUDE]
        ,[TRANSFORMERLONGITUDE] AS XFMRLONGITUDE
        ,[TRANSFORMERLATITUDE] AS XFMRLATITUDE
        ,[METERTYPE]
        ,[EUI]
        ,[PHASE]
        ,[ADDRESS]
        ,[SUBSTATION]
        ,[STREETVIEW]
        ,[POLE] AS XFMRPOLE
        ,[CIRCUIT]
        ,[FACILITYID] AS XFMRFACID
        ,[SIZE] AS XFMRSIZE
        ,[MAKE] AS XFMRMAKE
        ,[PRIMVOLT] AS XFMRPRIMVOLT
        ,[SECVOLT] AS XFMRSECVOLT
        ,[INSTALLDATE] AS XFMRINSTALLDATE
        ,[STOCKNUM] AS XFMRSTOCKNUM
        ,[XFMRTYPE]
        ,[RATEDKVA] AS XFMRRATEDKVA
    FROM [GridAnalysis].[dbo].[MetersTransformers]
    """
    conn.add_output_converter(-151, _handle_geometry)
    results = cursor.execute(meters_xfmr_query).fetchall()
    meters_xfmrs = pd.read_sql_query(meters_xfmr_query, conn)
    return meters_xfmrs
    

def get_xfmrs_info(conn):
    cursor = conn.cursor()
    xfmrs_query = """
    SELECT [LOCATION]
        ,[POLE]
        ,[CIRCUIT]
        ,[INYARD]
        ,[FACILITYID] AS XFMRFACID
        ,[SERIALNUMBER]
        ,[SIZE] AS XFMRSIZE
        ,[MAKE] AS XFMRMAKE
        ,[PRIMVOLT]
        ,[SECVOLT]
        ,[INSTALLDATE]
        ,[STOCKNUM]
        ,[XFMRTYPE]
        ,[RATEDKVA]
        ,[LONGITUDE]
        ,[LATITUDE]
    FROM [GridAnalysis].[dbo].[Transformers]
    """
    results = cursor.execute(xfmrs_query).fetchall()
    xfmrs = pd.read_sql_query(xfmrs_query, conn)
    #xfmrs["XFMRSIZE"] = pd.to_numeric(xfmrs["XFMRSIZE"], errors="coerce")
    transformer = _transform_coords()
    xfmrs["LONGITUDE"], xfmrs["LATITUDE"] = transformer.transform(xfmrs["LONGITUDE"], xfmrs["LATITUDE"])
    return xfmrs

def _transform_coords():
    return Transformer.from_crs("EPSG:2249", "EPSG:4326", always_xy=True)

def get_reads_by_date(start_date, end_date, conn):
    query = F"""
    DECLARE @START_DATE AS DATETIME
    DECLARE @END_DATE AS DATETIME
    SET @START_DATE = '{start_date}'
    SET @END_DATE = '{end_date}'
    SELECT * FROM [GridAnalysis].[dbo].[MonthlyKVAReads]
    WHERE [time] BETWEEN @START_DATE AND @END_DATE
    ORDER BY [time];
    """
    #print(f"Fetching reads from {start_date} to {end_date}...")
    reads = pd.read_sql_query(query, conn)
    reads["time"] = pd.to_datetime(reads["time"])
    reads["time"] = reads["time"].dt.tz_localize("America/New_York")
    #print("Reads fetched!")
    return reads


# -----------------------------
# Data prep / circuit helpers
# -----------------------------
def get_unique_circuits(xfmrs):
    return sorted(pd.unique(xfmrs["CIRCUIT"].dropna().astype(str)))

def get_xfmrs_by_circuit(xfmrs, circuit):
    return xfmrs[xfmrs["CIRCUIT"].astype(str).str.strip().eq(str(circuit))].copy()

def get_xfmr_location(fac_id, xfmrs_df):
    if not isinstance(fac_id, str):
        fac_id_str = str(fac_id)
    else:
        fac_id_str = fac_id

    known_ids = sorted(pd.unique(xfmrs_df["XFMRFACID"].astype(str)))
    if fac_id_str not in known_ids:
        return None

    row = xfmrs_df[xfmrs_df["XFMRFACID"].astype(str) == fac_id_str]
    if row.empty:
        return None

    return {
        "LONGITUDE": row["LONGITUDE"].item(),
        "LATITUDE": row["LATITUDE"].item(),
    }

READS_CACHE = {}
WEATHER_CACHE = {}
METER_CACHE = {}

last_date_params = {"start": None, "end": None}

def dates_changed():
    """Check if dates have changed since last refresh"""
    current_start = start_date_input.value.strftime('%Y-%m-%d')
    current_end = end_date_input.value.strftime('%Y-%m-%d')
    
    changed = (current_start != last_date_params["start"] or 
               current_end != last_date_params["end"])
    
    if changed:
        last_date_params["start"] = current_start
        last_date_params["end"] = current_end
    
    return changed

def cache_df(cache_dict, key_builder):
    def decorator(func):
        def wrapper(*args, **kwargs):
            key = key_builder(*args, **kwargs)
            if key not in cache_dict:
                cache_dict[key] = func(*args, **kwargs)
            return cache_dict[key].copy()
        return wrapper
    return decorator

@cache_df(READS_CACHE, lambda start_date, end_date, conn: (start_date, end_date, conn))
def cached_reads(start_date, end_date, conn):
    return get_reads_by_date(start_date, end_date, conn)

@cache_df(METER_CACHE, lambda conn: conn)
def cached_meters(conn):
    return get_meters_xfmrs(conn)

@cache_df(WEATHER_CACHE, lambda start_date, end_date: (start_date, end_date))
def cached_weather(start_date, end_date):
    return get_weather(start_date, end_date)

# -----------------------------
# Load / weather functions
# -----------------------------
def get_weather(start_date, end_date):

	# Set up the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
	retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
	openmeteo = openmeteo_requests.Client(session=retry_session)

	# API config
	url = "https://api.open-meteo.com/v1/forecast"
	params = {
		"latitude": 42.499683,
		"longitude": -70.863861,
		"hourly": "temperature_2m",
		"timezone": "America/New_York",
		"wind_speed_unit": "mph",
		"temperature_unit": "fahrenheit",
		"precipitation_unit": "inch",
		"start_date": start_date,
		"end_date": end_date,
	}
	responses = openmeteo.weather_api(url, params = params)

	# Process response
	response = responses[0]
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

	hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
			end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		).tz_convert(response.Timezone().decode())
	}

	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_df = pd.DataFrame(data = hourly_data)
	#print("\nHourly data\n", hourly_df)
	return hourly_df

def get_hourly_xfmr_load(meters_xfmrs, reads, circuit=None):
    

    reads = reads.copy()
    meters_xfmrs = meters_xfmrs.copy()

    reads["meter_id"] = reads["meter_id"].astype("str")
    meters_xfmrs["METERID"] = meters_xfmrs["METERID"].astype("str")
    reads["time"] = pd.to_datetime(reads["time"])

    if circuit: # select by circuit if provided
        circuit_mapping = get_xfmrs_by_circuit(meters_xfmrs, circuit)
        reads = reads.merge(
            circuit_mapping[["METERID", "XFMRFACID", "XFMRSIZE"]],
            left_on="meter_id",
            right_on="METERID",
            how="inner",
        )

    hourly_xfmr_load = (
        reads.groupby(["XFMRFACID", "XFMRSIZE", pd.Grouper(key="time", freq="1h")])["kwh_usage"]
        .sum()
        .reset_index()
    )

    hourly_xfmr_load["kwh_usage"] = hourly_xfmr_load["kwh_usage"].astype(float)
    hourly_xfmr_load["XFMRSIZE"] = hourly_xfmr_load["XFMRSIZE"].astype(float)
    hourly_xfmr_load["pct_load"] = (hourly_xfmr_load["kwh_usage"] / hourly_xfmr_load["XFMRSIZE"]) * 100

    return hourly_xfmr_load

def match_xfmr_weather(hourly_xfmr_load, start_date, end_date):
    weather = get_weather(start_date, end_date)
    out = hourly_xfmr_load.merge(
        weather,
        left_on="time",
        right_on="date",
        how="inner"
    ).drop(columns=["date"])
    return out

def get_hot_hours(hourly_xfmr_load, temp_threshold=80.0):
    hot_hour_data = hourly_xfmr_load[hourly_xfmr_load["temperature_2m"] >= temp_threshold].copy()
    if hot_hour_data.empty:
        raise ValueError(f"No hot-hour rows found above {temp_threshold}°F.")
    return hot_hour_data.sort_values(["XFMRFACID", "time"]).reset_index(drop=True)

# -----------------------------
# Risk summary
# -----------------------------
def get_risk_summary(hot_hour_data, temp_threshold=80.0, load_threshold=125):
    if load_threshold_input.value is not 125:
        load_threshold = load_threshold_input.value
        
    risk_summary = (
        hot_hour_data.groupby("XFMRFACID")
        .agg(
            max_pct_load=("pct_load", "max"),
            avg_hot_temp_f=("temperature_2m", "mean"),
            hot_hours=("temperature_2m", lambda s: int((s >= temp_threshold).sum())),
            hours_over_load_threshold=("pct_load", lambda s: int((s > load_threshold).sum()))
        )
        .reset_index()
        .sort_values(["max_pct_load", "hours_over_load_threshold"], ascending=False)
        .reset_index(drop=True)
    )
    if risk_summary.empty:
        raise ValueError(f"No transformers over {load_threshold}% load.")
    return risk_summary

# -----------------------------
# Risk map helpers
# -----------------------------
def get_risk_map_df(hot_hour_data, xfmrs_df, circuit_names=None):
    if circuit_names is None:
        circuit_names = get_unique_circuits(xfmrs_df)

    circuit_names = [str(c) for c in circuit_names]

    risk_summary = get_risk_summary(hot_hour_data, temp_threshold=50.0)

    # --- HANDLE INFINITIES & CREATE HOVER TEXT ---
    if "max_pct_load" in risk_summary.columns:
        # Replace inf with NaN for the numeric calculations/color scale
        risk_summary["max_pct_load_numeric"] = risk_summary["max_pct_load"].replace([np.inf, -np.inf], np.nan)
        
        # Create a user-friendly string column for the hover tooltip
        def format_pct_load(val):
            if pd.isna(val) or np.isinf(val):
                return "Invalid Data (Check Rating)"
            return f"{val:.1f}%"
            
        risk_summary["max_pct_load_hover"] = risk_summary["max_pct_load"].apply(format_pct_load)
    else:
        risk_summary["max_pct_load_numeric"] = np.nan
        risk_summary["max_pct_load_hover"] = "N/A"

    risk_summary = risk_summary.merge(
        xfmrs_df[["XFMRFACID", "CIRCUIT", "LATITUDE", "LONGITUDE"]].drop_duplicates(),
        on="XFMRFACID",
        how="left"
    )

    mapped_df = risk_summary[risk_summary["CIRCUIT"].astype(str).isin(circuit_names)].copy()
    mapped_df = mapped_df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
    mapped_df["LATITUDE"] = pd.to_numeric(mapped_df["LATITUDE"], errors="coerce")
    mapped_df["LONGITUDE"] = pd.to_numeric(mapped_df["LONGITUDE"], errors="coerce")
    mapped_df = mapped_df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

    mapped_df["marker_size"] = mapped_df["hours_over_load_threshold"].clip(lower=1, upper=20) * 6 + 8
    return mapped_df

def plot_risk_map(mapped_df, selected_circuits):
    if mapped_df.empty:
        raise ValueError("No mapped transformer points available for the selected circuits.")

    fig = px.scatter_map(
        mapped_df,
        lat="LATITUDE",
        lon="LONGITUDE",
        color="max_pct_load_numeric", # Used for continuous color scale
        hover_name="XFMRFACID",
        custom_data=["max_pct_load_hover", "avg_hot_temp_f", "hours_over_load_threshold", "CIRCUIT"],
        color_continuous_scale="YlOrRd",
        size_max=20,
        size="hours_over_load_threshold",
        zoom=13.5,
        center={
            "lat": mapped_df["LATITUDE"].mean(),
            "lon": mapped_df["LONGITUDE"].mean(),
        },
        opacity=0.9,
        map_style="open-street-map",
    )

    # Customize the hover template to show your custom text string
    fig.update_traces(
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "Circuit: %{customdata[3]}<br>"
            "Max % Load: <b>%{customdata[0]}</b><br>"
            "Avg Hot Temp (°F): %{customdata[1]:.1f}°F<br>"
            "Hours Over Threshold: %{customdata[2]}<br>"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        #title=f"Risky transformers during hot hours ({', '.join([str(c) for c in selected_circuits])})",
        title=f"Transformer Risk Map",
        title_x=0.0,
        title_xanchor="left",
        margin=dict(l=0, r=0, t=40, b=0),
        width=1500,
        height=map_h-40,
        map=dict(style="light")
    )

    return fig

# -----------------------------
# Heatmap helpers
# -----------------------------
def get_heatmap_data(hot_hour_data):
    return hot_hour_data.pivot_table(
        index="XFMRFACID",
        columns=hot_hour_data["time"].dt.strftime("%m-%d %H:00"),
        values="pct_load",
        aggfunc="mean"
    )

In [ ]:
# DASHBOARD
# ==========================================
max_limit = 100.0
first_limit = 80.0
N_DAYS_AGO = 90
tod = datetime.now()
d = timedelta(days = N_DAYS_AGO)
d1 = timedelta(days = N_DAYS_AGO-1)
#default_start_date = (tod - d)
#default_end_date = (tod - d1)
default_start_date = pd.to_datetime('2026-07-01')
default_end_date = pd.to_datetime('2026-07-02')
#print(f"{default_start_date} - {default_end_date}")

server = "MMLDAPP03"
database = "GridAnalysis"
driver = "{ODBC Driver 17 for SQL Server}"

from ipywidgets import widgets, Layout
import pandas as pd
from IPython.display import clear_output, HTML, display

from ipywidgets import widgets, Layout

def status_message(msg, bg="#e0f2fe", border="#7dd3fc"):
    return widgets.HTML(
        value=f"""
        <div style="
            font-family: 'Segoe UI', sans-serif;
            font-size: 14px;
            padding: 16px 20px;
            border-radius: 10px;
            background: {bg};
            border: 1px solid {border};
            color: #1f2937;
            width: 100%;
            height: 100%;
            display: flex;
            align-items: center;
            justify-content: center;
            text-align: center;
            box-sizing: border-box;
        ">
            ⚠️ &nbsp; {msg}
        </div>
        """
    )

# --- Styling ---
panel_style = {
    "border": "1px solid #dfe6ee",
    "border_radius": "12px",
    "padding": "14px 16px",
    "background": "#f8fafc",
    "box_shadow": "0 1px 2px rgba(15,23,42,0.04)",
}

section_title_style = (
    "font-size: 13px; font-weight: 700; color: #1f2937; "
    "margin: 0 0 8px 0; letter-spacing: 0.02em;"
)

legend_chip = (
    "display:inline-block; padding:4px 8px; border-radius:999px; "
    "font-size:11px; font-weight:600; color:#111827; "
    "margin-right:8px; margin-bottom:6px; "
    "border:1px solid rgba(0,0,0,0.08);"
)

instructions_header = widgets.HTML(
    value="""
    <div style="
        width: 100%;
        background: linear-gradient(135deg, #0f172a, #1e293b);
        color: white;
        font-weight: 700;
        font-size: 15px;
        padding: 10px 14px;
        border-radius: 12px 12px 0 0;
        box-sizing: border-box;
    ">
        Instructions
    </div>
    """
)

instructions_html = f"""
<div style="font-family: 'Segoe UI', sans-serif; line-height:1.55; color:#1f2937;">
    <div style="font-size: 17px; font-weight:700; margin-bottom:10px; color:#0f172a;">
        Dashboard Instructions
    </div>

    <div style="{section_title_style}">How to use this tool</div>
    <ul style="margin: 0 0 12px 18px; padding:0;">
        <li>Choose a date window using <b>Start</b> and <b>End</b>.</li>
        <li>Adjust <b>Temp Threshold</b> to define which hours count as hot periods.</li>
        <li>Adjust <b>Max Load</b> to identify transformers exceeding a percent load threshold.</li>
        <li>Select one or more circuits from the list, then click <b>Refresh</b>.</li>
    </ul>

    <div style="{section_title_style}">How to read the map</div>
    <ul style="margin: 0 0 12px 18px; padding:0;">
        <li>Each marker is a transformer.</li>
        <li>Marker <b>color</b> reflects the transformer’s <b>maximum observed % load</b> during hot hours.</li>
        <li>Marker <b>size</b> reflects how many hot hours exceeded the selected load threshold.</li>
        <li>Red/orange tones usually indicate higher risk or higher peak loading.</li>
    </ul>

    <div style="{section_title_style}">Legend</div>
    <div style="margin-bottom: 10px;">
        <span style="{legend_chip}; background:#fff7ed; border-color:#fdba74;">High risk</span>
        <span style="{legend_chip}; background:#fee2e2; border-color:#fca5a5;">Lower risk</span>
        <span style="{legend_chip}; background:#f3f4f6; border-color:#d1d5db;">Invalid</span>
    </div>
</div>
"""

instructions_box = widgets.HTML(
    value=instructions_html,
    layout=Layout(
        width="100%",
        padding="14px 16px",
        border="1px solid #dfe6ee",
        border_top="none",
        border_radius="0 0 12px 12px",
        background="#f8fafc",
        box_shadow="0 1px 3px rgba(15, 23, 42, 0.06)",
        margin="0 0 15px 0"
    )
)

instructions_card = widgets.VBox(
    [instructions_header, instructions_box],
    layout=widgets.Layout(width="100%", margin="0px")
)

# Expanded widths to prevent horizontal text clipping/scrollbars
label_w = "140px"
input_w = "300px"
row_w = "480px"
selector_w = "300px"

def make_row(label_text, widget, row_width="435px"):
    return widgets.HBox(
        [
            widgets.Label(
                value=label_text,
                layout=widgets.Layout(
                    width=label_w,
                    justify_content="flex-end",
                    display="flex",
                    margin="0 8px 0 0"
                )
            ),
            widget
        ],
        layout=widgets.Layout(
            display="flex",
            align_items="center",
            justify_content="flex-start",
            width=row_width,
            margin="0 0 8px 0"
        )
    )
# RUNNING

conn = connect_to_db()
xfmrs = get_xfmrs_info(conn)

# params inputs with DatePicker
start_date_input = widgets.DatePicker(
    value=default_start_date,
    description='',
    layout=widgets.Layout(width=input_w)
)

end_date_input = widgets.DatePicker(
    value=default_end_date,
    description='',
    layout=widgets.Layout(width=input_w)
)

temp_threshold_input = widgets.FloatSlider(
    value=50.0,
    min=30.0,
    max=100.0,
    step=1.0,
    description='',
    layout=widgets.Layout(width=input_w),
    continuous_update = False
)

load_threshold_input = widgets.FloatSlider(
    value=125.0,
    min=100.0,
    max=200.0,
    step=1.0,
    description='',
    layout=widgets.Layout(width=input_w),
    continuous_update = False
)

params_form = widgets.VBox([
    make_row("Start", start_date_input),
    make_row("End", end_date_input),
    make_row("Temp Threshold (°F)", temp_threshold_input),
    make_row("Max Load (%)", load_threshold_input),
], layout=widgets.Layout(
    align_items="flex-start",
    margin="0 0 10px 0"
))
params_widget_list = [start_date_input, end_date_input, temp_threshold_input, load_threshold_input]

all_circuits = get_unique_circuits(xfmrs)
default_circuit = "CREESY" if "CREESY" in all_circuits else all_circuits[1]

circuit_selector = widgets.SelectMultiple(
    options=all_circuits,
    value=[default_circuit],
    description="",
    layout=widgets.Layout(width=selector_w, height="140px")
)
selector_row = make_row("Circuits", circuit_selector, row_width=row_w)

button_all = widgets.Button(description="Select all")
button_clear = widgets.Button(description="Clear")
button_refresh = widgets.Button(description="Refresh")
out = widgets.Output()

# dev
global map_df

# Fixed height matching the left control panel perfectly
map_h = 845

out = widgets.Output(
    layout=Layout(
        flex="1",
        height=f"{map_h}px",
        min_height=f"{map_h}px",
        max_height=f"{map_h}px",
        border="1px solid #dfe6ee",
        border_radius="12px",
        overflow="auto",
        background="#ffffff",
        padding="10px"
    )
)

def refresh_map(change=None):
    with out:
        out.clear_output(wait=True)
        selected = list(circuit_selector.value)
        
        start = start_date_input.value.strftime('%Y-%m-%d')
        end = end_date_input.value.strftime('%Y-%m-%d')
        temp_thresh = temp_threshold_input.value
        
        if not selected:
            display(status_message("No circuits selected. Please choose at least one circuit.", "#fef3c7", "#fbbf24"))
            return

        try:
            reads = cached_reads(start, end, conn)
            meter_xfmrs = cached_meters(conn)

            frames = []
            for circuit in selected:
                hourly = get_hourly_xfmr_load(meter_xfmrs, reads, circuit)
                
                #display(hourly[hourly["XFMRFACID"]==1182]) # DEV
                if hourly.empty:
                    continue
                frames.append(match_xfmr_weather(hourly, start, end))

            if not frames:
                display(status_message("No transformer load data found for the selected circuits.", "#fef2f2", "#fca5a5"))
                return

            hot = get_hot_hours(pd.concat(frames, ignore_index=True), temp_threshold=temp_thresh)
            map_df = get_risk_map_df(hot, xfmrs, circuit_names=selected)

            if map_df.empty:
                display(status_message("No risky transformers found for the selected circuits and parameters.", "#fef2f2", "#fca5a5"))
                return

            map_df = map_df[map_df["hours_over_load_threshold"] != 0]
            
            display(plot_risk_map(map_df, selected))

        except Exception as exc:
            display(status_message(f"Map update failed: {exc}<br>Try again with different parameters.", "#fef2f2", "#fca5a5"))

def select_all(_):
    circuit_selector.value = tuple(all_circuits)

def clear_all(_):
    circuit_selector.value = ()

for w in params_widget_list:
    w.observe(refresh_map)

button_all.on_click(select_all)
button_clear.on_click(clear_all)
button_refresh.on_click(refresh_map)
circuit_selector.observe(refresh_map, names="value")
# TODO: Add input for time over limit threshold

circuit_buttons = widgets.HBox(
    [button_all, button_clear, button_refresh],
    layout=widgets.Layout(
        display="flex",
        justify_content="space-between",
        width=selector_w,
        margin="0 10px 0 148px"
    )
)

circuit_section = widgets.VBox(
    [selector_row, circuit_buttons],
    layout=widgets.Layout(
        align_items="flex-start",
        margin="0 0 0 -15px"
    )
)

main_text_box = widgets.VBox(
    [instructions_card, params_form, circuit_section],
    layout=widgets.Layout(
        width="500px",
        min_width="500px",
        min_height=f"{map_h}px",
        margin="0 10px 0 0"
    )
)

main_box = widgets.HBox(
    [main_text_box, out],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        padding="10px"
    )
)

# TODO: Add section for current inputs/selections on map to make clear for exports
display(main_box)
refresh_map()